# 📊 Retail Sales Analysis - End-to-End Data Analytics Portfolio Project
**Author:** Data Analyst Portfolio Project  
**Target Role:** Junior Data Analyst / Data Analytics Intern  
**Tech Stack:** Python, Pandas, NumPy, Matplotlib, Seaborn, Power BI  

---

## 📌 1. Project Overview & Business Problem
In the modern omnichannel retail sector, business sustainability depends on balancing **revenue growth** with **healthy profit margins**. Retail businesses frequently experience disconnects between top-line sales figures and actual profitability due to:
1. Aggressive or uncalibrated promotional discount depths.
2. High product return / cancellation rates.
3. Disparities between high-volume, low-margin categories (e.g., Electronics) vs. low-volume, high-margin categories (e.g., Beauty, Clothing).
4. Regional variations in customer purchasing power and Average Order Value (AOV).

### 🎯 Project Objectives:
- Ingest and audit a realistic retail transactions dataset containing genuine data quality issues.
- Clean and standardize the data (handling duplicates, missing values, string anomalies, and type conversions).
- Engineer critical financial and calendar metrics (`Gross_Sales`, `Discount_Amount`, `Net_Sales`, `Total_Cost`, `Profit`, `Profit_Margin_Pct`, date dimensions).
- Compute Executive KPIs (Total Sales, Total Profit, Profit Margin %, Total Orders, AOV).
- Analyze temporal trends (monthly seasonality and MoM growth rates).
- Evaluate category and geographic performance.
- Isolate top-performing and underperforming/loss-making products to provide actionable business recommendations.
- Prepare a normalized, clean dataset ready for interactive Power BI dashboards.

*Note: All data analyzed is synthetic and simulated for educational and portfolio demonstration purposes.*


---
## 🛠️ 2. Environment Setup & Data Loading
We start by importing our essential data manipulation and visualization libraries, configuring standard aesthetic parameters.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configure display options and plot styling
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial', 'DejaVu Sans', 'Helvetica'
plt.rcParams['figure.dpi'] = 120

print("Libraries imported successfully!")


### 2.1 Load Raw Dataset & Inspect Structure


In [ ]:
raw_filepath = "../data/raw/retail_sales_raw.csv"
if not os.path.exists(raw_filepath):
    raw_filepath = "data/raw/retail_sales_raw.csv"

df_raw = pd.read_csv(raw_filepath)
print(f"Raw Dataset Shape: {df_raw.shape[0]:,} Rows, {df_raw.shape[1]} Columns")
df_raw.head()


### 2.2 Initial Data Audit: Information & Missing Values


In [ ]:
# Data types and non-null counts
df_raw.info()


In [ ]:
# Check missing values per column
missing_summary = df_raw.isnull().sum()
missing_summary[missing_summary > 0]


In [ ]:
# Check duplicate records
print(f"Total Duplicate Rows: {df_raw.duplicated().sum()}")


---
## 🧹 3. Data Cleaning & Preprocessing
Real-world data is messy. In this section, we apply rigorous cleaning steps:
1. **Deduplication:** Remove exact and Transaction_ID duplicate rows.
2. **Text Normalization:** Trim extra whitespace and harmonize letter casing in categorical columns.
3. **Numeric Type Conversion:** Strip currency symbols (`$`) and commas from price columns.
4. **Invalid Value Handling:** Remove non-positive quantities (`Quantity <= 0`).
5. **Missing Value Resolution:** Parse mixed-format dates and drop unrecoverable records; impute categorical nulls with contextual defaults.
6. **Feature Engineering:** Calculate financial metrics and calendar attributes.


In [ ]:
# Step 1: Remove duplicates
df_clean = df_raw.drop_duplicates().copy()
df_clean = df_clean.drop_duplicates(subset=['Transaction_ID'], keep='first').copy()
print(f"Rows after duplicate removal: {len(df_clean):,}")


In [ ]:
# Step 2: Clean text columns
text_cols = ['Customer_Segment', 'Region', 'Product_Category', 'Product_Subcategory', 
             'Product_Name', 'Payment_Method', 'Shipping_Status']
for col in text_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].astype(str).str.strip().replace({'nan': np.nan, 'None': np.nan})

df_clean['Region'] = df_clean['Region'].str.title()
df_clean['Product_Category'] = df_clean['Product_Category'].str.strip()


In [ ]:
# Step 3: Standardize numeric columns
if 'Unit_Price' in df_clean.columns:
    df_clean['Unit_Price'] = (
        df_clean['Unit_Price'].astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
    )
    df_clean['Unit_Price'] = pd.to_numeric(df_clean['Unit_Price'], errors='coerce')

df_clean['Unit_Cost'] = pd.to_numeric(df_clean['Unit_Cost'], errors='coerce')
df_clean['Quantity'] = pd.to_numeric(df_clean['Quantity'], errors='coerce')
df_clean['Discount'] = pd.to_numeric(df_clean['Discount'], errors='coerce').fillna(0.0)

# Filter out non-positive quantities
df_clean = df_clean[df_clean['Quantity'] > 0].copy()
df_clean['Quantity'] = df_clean['Quantity'].astype(int)
print(f"Rows after filtering non-positive quantities: {len(df_clean):,}")


In [ ]:
# Step 4: Resolve missing dates & categories
df_clean['Order_Date'] = pd.to_datetime(df_clean['Order_Date'], format='mixed', errors='coerce')
df_clean = df_clean.dropna(subset=['Order_Date']).copy()

df_clean['Customer_Segment'] = df_clean['Customer_Segment'].fillna('Consumer')
df_clean['Payment_Method'] = df_clean['Payment_Method'].fillna('Not Specified')
print(f"Final Cleaned Rows: {len(df_clean):,}")


### 3.1 Feature Engineering: Financials & Date Hierarchy


In [ ]:
# Financial Calculations
df_clean['Gross_Sales'] = (df_clean['Quantity'] * df_clean['Unit_Price']).round(2)
df_clean['Discount_Amount'] = (df_clean['Gross_Sales'] * df_clean['Discount']).round(2)
df_clean['Net_Sales'] = (df_clean['Gross_Sales'] - df_clean['Discount_Amount']).round(2)
df_clean['Total_Cost'] = (df_clean['Quantity'] * df_clean['Unit_Cost']).round(2)
df_clean['Profit'] = (df_clean['Net_Sales'] - df_clean['Total_Cost']).round(2)
df_clean['Profit_Margin_Pct'] = np.where(
    df_clean['Net_Sales'] > 0,
    ((df_clean['Profit'] / df_clean['Net_Sales']) * 100).round(2),
    0.0
)

# Margin Classification
conditions = [
    (df_clean['Profit_Margin_Pct'] >= 25.0),
    (df_clean['Profit_Margin_Pct'] >= 10.0) & (df_clean['Profit_Margin_Pct'] < 25.0),
    (df_clean['Profit_Margin_Pct'] >= 0.0) & (df_clean['Profit_Margin_Pct'] < 10.0),
    (df_clean['Profit_Margin_Pct'] < 0.0)
]
choices = ['High Margin (>=25%)', 'Moderate Margin (10-25%)', 'Low Margin (0-10%)', 'Loss Making (<0%)']
df_clean['Margin_Category'] = np.select(conditions, choices, default='Unclassified')

# Date Dimensions
df_clean['Year'] = df_clean['Order_Date'].dt.year
df_clean['Month'] = df_clean['Order_Date'].dt.month
df_clean['Month_Name'] = df_clean['Order_Date'].dt.strftime('%b')
df_clean['Quarter'] = 'Q' + df_clean['Order_Date'].dt.quarter.astype(str)
df_clean['Day_of_Week'] = df_clean['Order_Date'].dt.day_name()
df_clean['Year_Month'] = df_clean['Order_Date'].dt.strftime('%Y-%m')

df_clean = df_clean.sort_values(by='Order_Date').reset_index(drop=True)
df_clean.head()


---
## 📈 4. Exploratory Data Analysis & KPI Computation
We compute high-level business metrics to assess operational scale, profitability, and customer behavior.


In [ ]:
# Executive KPIs
total_sales = df_clean['Net_Sales'].sum()
total_gross = df_clean['Gross_Sales'].sum()
total_profit = df_clean['Profit'].sum()
total_orders = df_clean['Transaction_ID'].nunique()
total_units = df_clean['Quantity'].sum()
aov = total_sales / total_orders
profit_margin = (total_profit / total_sales) * 100
avg_discount = df_clean['Discount'].mean() * 100

kpi_summary = pd.DataFrame({
    'Metric': [
        'Total Net Sales Revenue', 'Total Gross Sales', 'Total Net Profit',
        'Overall Profit Margin', 'Total Orders', 'Total Units Sold',
        'Average Order Value (AOV)', 'Average Discount Rate'
    ],
    'Value': [
        f"${total_sales:,.2f}", f"${total_gross:,.2f}", f"${total_profit:,.2f}",
        f"{profit_margin:.2f}%", f"{total_orders:,}", f"{total_units:,}",
        f"${aov:,.2f}", f"{avg_discount:.2f}%"
    ]
})
kpi_summary


### 4.1 Monthly Sales & Profit Trend Analysis
Let's examine seasonal purchase patterns, peak months, and Month-over-Month (MoM) revenue growth.


In [ ]:
monthly_trend = (
    df_clean.groupby(['Year', 'Month', 'Month_Name', 'Year_Month'])
    .agg(
        Net_Sales=('Net_Sales', 'sum'),
        Profit=('Profit', 'sum'),
        Orders=('Transaction_ID', 'count'),
        Units_Sold=('Quantity', 'sum')
    )
    .reset_index()
    .sort_values(by=['Year', 'Month'])
)
monthly_trend['MoM_Sales_Growth_Pct'] = monthly_trend['Net_Sales'].pct_change() * 100
monthly_trend['Profit_Margin_Pct'] = (monthly_trend['Profit'] / monthly_trend['Net_Sales']) * 100
monthly_trend['AOV'] = monthly_trend['Net_Sales'] / monthly_trend['Orders']
monthly_trend


In [ ]:
# Visualization 1: Monthly Sales & Profit Trend
fig, ax1 = plt.subplots(figsize=(12, 5.5))

color_sales = '#1f77b4'
ax1.plot(monthly_trend['Month_Name'], monthly_trend['Net_Sales'] / 1000, 
         marker='o', linewidth=2.5, color=color_sales, label='Net Sales ($K)')
ax1.fill_between(monthly_trend['Month_Name'], monthly_trend['Net_Sales'] / 1000, color=color_sales, alpha=0.1)
ax1.set_ylabel('Net Sales ($ in Thousands)', color=color_sales, fontsize=11, fontweight='bold')
ax1.tick_params(axis='y', labelcolor=color_sales)

ax2 = ax1.twinx()
color_profit = '#2ca02c'
ax2.plot(monthly_trend['Month_Name'], monthly_trend['Profit'] / 1000, 
         marker='s', linewidth=2.2, linestyle='--', color=color_profit, label='Net Profit ($K)')
ax2.set_ylabel('Net Profit ($ in Thousands)', color=color_profit, fontsize=11, fontweight='bold')
ax2.tick_params(axis='y', labelcolor=color_profit)

plt.title('Monthly Sales & Profit Performance (2023)', fontsize=14, fontweight='bold', pad=15)
ax1.grid(True, linestyle=':', alpha=0.6)
fig.tight_layout()
plt.show()


### 4.2 Product Category Performance Breakdown
Evaluating which product categories generate the highest revenue vs. which ones yield the highest margins.


In [ ]:
cat_summary = (
    df_clean.groupby('Product_Category')
    .agg(
        Net_Sales=('Net_Sales', 'sum'),
        Profit=('Profit', 'sum'),
        Orders=('Transaction_ID', 'count'),
        Units_Sold=('Quantity', 'sum'),
        Avg_Discount=('Discount', lambda x: x.mean() * 100)
    )
    .reset_index()
)
cat_summary['Profit_Margin_Pct'] = (cat_summary['Profit'] / cat_summary['Net_Sales']) * 100
cat_summary['Revenue_Share_Pct'] = (cat_summary['Net_Sales'] / total_sales) * 100
cat_summary = cat_summary.sort_values(by='Net_Sales', ascending=False).reset_index(drop=True)
cat_summary


In [ ]:
# Visualization 2: Category Net Sales vs. Profit
fig, ax = plt.subplots(figsize=(10, 5))
y_pos = np.arange(len(cat_summary))
height = 0.35

ax.barh(y_pos + height/2, cat_summary['Net_Sales']/1000, height, label='Net Sales ($K)', color='#2b5c8f')
ax.barh(y_pos - height/2, cat_summary['Profit']/1000, height, label='Net Profit ($K)', color='#46a066')

ax.set_yticks(y_pos)
ax.set_yticklabels(cat_summary['Product_Category'], fontsize=10, fontweight='bold')
ax.set_xlabel('Amount ($ in Thousands)', fontsize=11, fontweight='bold')
ax.set_title('Category Comparison: Net Sales vs. Profit', fontsize=13, fontweight='bold', pad=12)
ax.legend()
ax.grid(axis='x', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


### 4.3 Geographic & Regional Analysis
Analyzing regional sales contribution, profit margins, and Average Order Values (AOV) across geographical territories.


In [ ]:
reg_summary = (
    df_clean.groupby('Region')
    .agg(
        Net_Sales=('Net_Sales', 'sum'),
        Profit=('Profit', 'sum'),
        Orders=('Transaction_ID', 'count'),
        AOV=('Net_Sales', 'mean')
    )
    .reset_index()
)
reg_summary['Profit_Margin_Pct'] = (reg_summary['Profit'] / reg_summary['Net_Sales']) * 100
reg_summary['Sales_Share_Pct'] = (reg_summary['Net_Sales'] / total_sales) * 100
reg_summary = reg_summary.sort_values(by='Net_Sales', ascending=False).reset_index(drop=True)
reg_summary


In [ ]:
# Visualization 3: Regional Sales & Profit Margins
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

ax1.bar(reg_summary['Region'], reg_summary['Net_Sales']/1000, color='#1f77b4', edgecolor='#0e4b75')
ax1.set_title('Total Net Sales by Region ($K)', fontweight='bold')
ax1.set_ylabel('Sales ($ in Thousands)')
for i, v in enumerate(reg_summary['Net_Sales']/1000):
    ax1.text(i, v + 2, f"${v:.1f}K", ha='center', fontweight='bold', fontsize=9)

ax2.bar(reg_summary['Region'], reg_summary['Profit_Margin_Pct'], color='#2ca02c', edgecolor='#1b631b')
ax2.set_title('Profit Margin by Region (%)', fontweight='bold')
ax2.set_ylabel('Margin (%)')
for i, v in enumerate(reg_summary['Profit_Margin_Pct']):
    ax2.text(i, v + 0.3, f"{v:.1f}%", ha='center', fontweight='bold', fontsize=9)

plt.tight_layout()
plt.show()


### 4.4 Top-Performing & Underperforming Products
Identifying our champion revenue and profit drivers versus loss-making inventory items.


In [ ]:
prod_analysis = (
    df_clean.groupby(['Product_Category', 'Product_Subcategory', 'Product_Name'])
    .agg(
        Net_Sales=('Net_Sales', 'sum'),
        Profit=('Profit', 'sum'),
        Units_Sold=('Quantity', 'sum'),
        Orders=('Transaction_ID', 'count'),
        Avg_Discount=('Discount', lambda x: x.mean() * 100)
    )
    .reset_index()
)
prod_analysis['Profit_Margin_Pct'] = (prod_analysis['Profit'] / prod_analysis['Net_Sales']) * 100

top_10_revenue = prod_analysis.sort_values(by='Net_Sales', ascending=False).head(10).reset_index(drop=True)
top_10_profit = prod_analysis.sort_values(by='Profit', ascending=False).head(10).reset_index(drop=True)
bottom_5_loss = prod_analysis.sort_values(by='Profit', ascending=True).head(5).reset_index(drop=True)

print("--- TOP 5 REVENUE DRIVERS ---")
display(top_10_revenue.head(5))

print("
--- BOTTOM 5 PRODUCTS BY PROFIT ---")
display(bottom_5_loss)


In [ ]:
# Visualization 4: Top vs Bottom Products
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Top 5 Revenue
ax1.barh(top_10_revenue.head(5)['Product_Name'], top_10_revenue.head(5)['Net_Sales']/1000, color='#1f77b4')
ax1.invert_yaxis()
ax1.set_title('Top 5 Products by Revenue ($K)', fontweight='bold')
ax1.set_xlabel('Sales ($ in Thousands)')

# Bottom 5 Profit
bar_colors = ['#d62728' if p < 0 else '#f39c12' for p in bottom_5_loss['Profit']]
ax2.barh(bottom_5_loss['Product_Name'], bottom_5_loss['Profit']/1000, color=bar_colors)
ax2.invert_yaxis()
ax2.set_title('Bottom 5 Products by Net Profit ($K)', fontweight='bold')
ax2.set_xlabel('Profit ($ in Thousands)')
ax2.axvline(0, color='black', linestyle='--')

plt.tight_layout()
plt.show()


---
## 💡 5. Business Insights & Strategic Recommendations

### 🔍 Key Findings:
1. **Electronics Dominates Top-Line but Moderates Margins:**
   - Electronics drives **61.1% of total sales revenue** ($489.5K) but operates at a moderate **20.8% profit margin**. Flagship devices (Smartphones, Laptops, 55" TVs) drive transaction volume and store traffic.
2. **Hidden Profit Stars in Beauty & Apparel:**
   - **Beauty & Personal Care** delivers the company's highest profit margin at **53.6%**, followed by **Clothing** at **46.8%**. While their revenue shares are smaller (5.2% and 5.4%), their margin efficiency makes them ideal for cross-selling and bundled offers.
3. **Discount Depth Erodes Clearance Profitability:**
   - The `Budget Graphic Polyester Tee` is our primary loss-making item, resulting in a net loss of `-$805.20` (-48.4% profit margin). Analysis reveals an excessive average discount rate of **35.6%**, which exceeds the baseline product markup.
4. **Geographic Consistency with High Central AOV:**
   - Regional demand is evenly distributed (North 21.4%, Central 20.1%, West 20.0%, East 19.2%, South 19.2%). The **Central Region** boasts the highest Average Order Value at **$312.22**, indicating strong appetite for higher-tier ticket items.
5. **Mid-Year Surge vs. Q4 Promotion Margin Compression:**
   - Sales peaked in August ($91.1K), while promotional holiday discounts in November/December reduced profit margins down to 22.9%.

### 🚀 Strategic Recommendations:
- **Cap Promotional Discounts on Low-Margin SKUs:** Set algorithmic discount ceilings on budget/clearance lines (maximum 15-20% discount) to prevent negative margins.
- **Bundle High-Margin Beauty/Accessories with Electronics:** Encourage checkout add-ons (cables, skincare, apparel accessories) with flagship laptop/phone purchases to elevate blended basket margin.
- **Capitalize on Central Region AOV:** Target Central customers with premium appliance and electronics bundles given their higher willingness to spend.
- **Re-engineer Holiday Discount Strategy:** Transition from blanket percentage discounts in Q4 to minimum-spend threshold promotions (e.g., "$25 off orders above $200") to protect holiday profit margins.


---
## 📊 6. Power BI Integration Readiness
The processed dataset `retail_sales_cleaned.csv` is completely normalized and ready for direct import into Microsoft Power BI:
- **Zero Nulls in Primary Keys:** Clean `Transaction_ID` and `Customer_ID`.
- **Pre-calculated Date Hierarchy:** `Year`, `Month`, `Month_Name`, `Quarter`, `Day_of_Week` for seamless slicers and time intelligence.
- **Calculated Measures Ready:** Pre-built `Net_Sales`, `Profit`, `Profit_Margin_Pct` matching DAX best practices.
- **Segment Filters:** `Customer_Segment`, `Region`, `Product_Category`, and `Margin_Category` ready for interactive cross-filtering.

*To import into Power BI: Open Power BI Desktop -> Get Data -> Text/CSV -> Select `data/processed/retail_sales_cleaned.csv` -> Load.*
